In [ ]:
import matplotlib.pyplot as plt
import os
import sys
import time
import socket
import threading
import csv
import numpy as np

from datetime import datetime
from ctypes import c_int16, c_int32, byref, c_char_p, c_float

from picosdk.ps6000 import ps6000 as ps
from picosdk.functions import adc2mV, assert_pico_ok


# ---------------------------------------------------------------------
# Local component imports
# ---------------------------------------------------------------------

if socket.gethostname() == "ph-photonbec5":
    sys.path.append(r"D:/Control/PythonPackages/")

import Components as comp


# ---------------------------------------------------------------------
# Hardware setup
# ---------------------------------------------------------------------

laser = comp.Toptica_Laser()

powermeter = comp.Thor_PowerMeter(
    power_meter_usb_name='USB0::0x1313::0x8078::P0034379::INSTR',
    num_power_readings=100,
    bs_factor=1,
    wavelength=950
)

def make_data_folder(base_directory='D:/Data'):
    year = datetime.now().strftime('%Y')
    year_month = datetime.now().strftime('%Y%m')
    year_month_day = datetime.now().strftime('%Y%m%d')

    folder_path = os.path.join(
        base_directory,
        year,
        year_month,
        year_month_day
    )

    os.makedirs(folder_path, exist_ok=True)
    return folder_path


def safe_number_string(x, ndigits=3):
    """
    Makes numbers filename-safe.

    Example:
        12.345 -> 12p345
        -1.2   -> m1p200
    """
    return f"{float(x):.{ndigits}f}".replace("-", "m").replace(".", "p")


def save_arrays(
        voltage_array,
        time_array,
        commanded_power_mW,
        measured_power_uW,
        iteration,
        base_directory='D:/Data',
        save_time_once=True
):
    folder_path = make_data_folder(base_directory)
    timestamp = datetime.now().strftime('%H%M%S')

    laserp_str = f"{commanded_power_mW:.3g}"
    measp_str = f"{measured_power_uW:.3g}"

    suffix = (
        f"laserp_{laserp_str}_mW"
        f"_measp_{measp_str}_uW"
        f"_iter{iteration:03d}"
    )

    voltage_filename = f"pbec_{timestamp}_{suffix}_Voltage.txt"
    voltage_path = os.path.join(folder_path, voltage_filename)

    np.savetxt(voltage_path, voltage_array, fmt='%0.4f')

    time_path = None

    if save_time_once:
        time_filename = "pbec_Time.txt"
        time_path = os.path.join(folder_path, time_filename)

        if not os.path.exists(time_path):
            np.savetxt(time_path, time_array, fmt='%0.4f')

    return voltage_path, time_path

# ---------------------------------------------------------------------
# PicoScope acquisition
# ---------------------------------------------------------------------

def get_picoscope_data(
        channels=[0],
        analogue_offset=0,
        trig=0,
        trigsource=0,
        trigthresh=64,
        trigdir=2,
        trigdelay=0,
        trigms=1000,
        preTriggerSamples=2500,
        postTriggerSamples=2500,
        timebase=8,
        live_plot=False
):
    status = {}
    chandle = c_int16()
    serial = c_char_p()

    status["openunit"] = ps.ps6000OpenUnit(byref(chandle), serial)
    assert status["openunit"] == 0, (
        f"Failed to open PicoScope 6407, "
        f"error code: {status['openunit']}"
    )

    try:
        channel_mapping = {
            0: 'PS6000_CHANNEL_A',
            1: 'PS6000_CHANNEL_B',
            2: 'PS6000_CHANNEL_C',
            3: 'PS6000_CHANNEL_D'
        }

        buffers = {}
        chARange = ps.PS6000_RANGE['PS6000_100MV']

        maxSamples = preTriggerSamples + postTriggerSamples

        for ch in channels:
            if ch not in channel_mapping:
                raise ValueError("Invalid channel. Use 0, 1, 2, or 3.")

            channel = ps.PS6000_CHANNEL[channel_mapping[ch]]
            enabled = 1
            coupling_type = ps.PS6000_COUPLING['PS6000_DC_50R']
            bandwidth = ps.PS6000_BANDWIDTH_LIMITER['PS6000_BW_FULL']

            status[f"set_channel_{ch}"] = ps.ps6000SetChannel(
                chandle,
                channel,
                enabled,
                coupling_type,
                chARange,
                analogue_offset,
                bandwidth
            )
            assert_pico_ok(status[f"set_channel_{ch}"])

            buffers[ch] = {
                'max': (c_int16 * maxSamples)(),
                'min': (c_int16 * maxSamples)()
            }

            status[f"setDataBuffers_{ch}"] = ps.ps6000SetDataBuffers(
                chandle,
                channel,
                byref(buffers[ch]['max']),
                byref(buffers[ch]['min']),
                maxSamples,
                0
            )
            assert_pico_ok(status[f"setDataBuffers_{ch}"])

        status["trigger"] = ps.ps6000SetSimpleTrigger(
            chandle,
            trig,
            trigsource,
            trigthresh,
            trigdir,
            trigdelay,
            trigms
        )
        assert_pico_ok(status["trigger"])

        timeIntervalns = c_float()
        returnedMaxSamples = c_int32()

        status["getTimebase2"] = ps.ps6000GetTimebase2(
            chandle,
            timebase,
            maxSamples,
            byref(timeIntervalns),
            1,
            byref(returnedMaxSamples),
            0
        )
        assert_pico_ok(status["getTimebase2"])

        status["runBlock"] = ps.ps6000RunBlock(
            chandle,
            preTriggerSamples,
            postTriggerSamples,
            timebase,
            0,
            None,
            0,
            None,
            None
        )
        assert_pico_ok(status["runBlock"])

        ready = c_int16(0)

        while ready.value == 0:
            status["isReady"] = ps.ps6000IsReady(chandle, byref(ready))

        overflow = c_int16()
        cmaxSamples = c_int32(maxSamples)

        status["getValues"] = ps.ps6000GetValues(
            chandle,
            0,
            byref(cmaxSamples),
            1,
            0,
            0,
            byref(overflow)
        )
        assert_pico_ok(status["getValues"])

        maxADC = c_int16(32512)

        channel_data = {}

        for ch in channels:
            adc2mV_buffer = adc2mV(
                buffers[ch]['max'],
                chARange,
                maxADC
            )
            channel_data[ch] = np.array(adc2mV_buffer)

        time_array = np.linspace(
            0,
            (cmaxSamples.value - 1) * timeIntervalns.value,
            cmaxSamples.value
        )

        if live_plot:
            plt.figure(figsize=(6, 4))

            for ch in channels:
                plt.plot(
                    time_array,
                    channel_data[ch],
                    label=f"Channel {channel_mapping[ch]}"
                )

            plt.xlabel('Time (ns)')
            plt.ylabel('Voltage (mV)')
            plt.title("PicoScope capture")
            plt.legend()
            plt.grid()
            plt.show()

        status["stop"] = ps.ps6000Stop(chandle)
        assert_pico_ok(status["stop"])

    finally:
        ps.ps6000CloseUnit(chandle)

    return channel_data, time_array


# ---------------------------------------------------------------------
# Power meter thread
# ---------------------------------------------------------------------

def power_meter_worker(
        powermeter,
        stop_event,
        measured_power,
        measured_time,
        dt=0.1
):
    t0 = time.time()

    while not stop_event.is_set():
        try:
            reading_uW = 1e6 * powermeter.take_power_reading()
            measured_power.append(reading_uW)
            measured_time.append(time.time() - t0)

        except Exception:
            pass

        time.sleep(dt)


def acquire_scope_and_power(
        powermeter,
        scope_kwargs,
        power_meter_dt=0.1
):
    measured_power = []
    measured_time = []

    t0 = time.time()

    #Take one immediate reading before scope acquisition
    try:
        first_reading_uW = 1e6 * powermeter.take_power_reading()
        measured_power.append(first_reading_uW)
        measured_time.append(time.time() - t0)
    except Exception as e:
        print("Initial power meter read failed:", e)

    stop_event = threading.Event()

    thread = threading.Thread(
        target=power_meter_worker,
        args=(
            powermeter,
            stop_event,
            measured_power,
            measured_time,
            power_meter_dt
        ),
        daemon=True
    )

    thread.start()

    try:
        channel_data, time_array = get_picoscope_data(**scope_kwargs)
    finally:
        stop_event.set()
        thread.join(timeout=2)

    # one final reading after scope acquisition
    try:
        final_reading_uW = 1e6 * powermeter.take_power_reading()
        measured_power.append(final_reading_uW)
        measured_time.append(time.time() - t0)
    except Exception as e:
        print("Final power meter read failed:", e)

    measured_power = np.array(measured_power)
    measured_time = np.array(measured_time)

    if len(measured_power) == 0:
        raise RuntimeError("No valid power meter readings were collected.")

    mean_power_uW = np.mean(measured_power)
    std_power_uW = np.std(measured_power)

    return (
        channel_data,
        time_array,
        measured_time,
        measured_power,
        mean_power_uW,
        std_power_uW
    )


# ---------------------------------------------------------------------
# Full power scan
# ---------------------------------------------------------------------

def run_power_scan(
        power_list,
        laser,
        powermeter,
        n_iterations=1,
        base_directory='D:/Data',
        settle_time_s=1.0,
        power_meter_dt=0.1,
        channel_to_save=0,
        voltage_threshold_mV=80,
        scope_kwargs=None
):
    if scope_kwargs is None:
        scope_kwargs = dict(
            channels=[0],
            analogue_offset=0,
            trig=0,
            trigsource=0,
            trigthresh=64,
            trigdir=2,
            trigdelay=0,
            trigms=1000,
            preTriggerSamples=2500,
            postTriggerSamples=2500,
            timebase=8,
            live_plot=True
        )

    iteration = 0

    for power_W in power_list:
        commanded_power_mW = power_W * 1e3

        print("")
        print(f"Setting laser to {commanded_power_mW:.4f} mW")

        laser.set(power_W)
        time.sleep(settle_time_s)

        for _ in range(n_iterations):
            print("")
            print(f"Iteration {iteration}")

            (
                channel_data,
                time_array,
                power_time,
                power_trace,
                mean_power_uW,
                std_power_uW
            ) = acquire_scope_and_power(
                powermeter=powermeter,
                scope_kwargs=scope_kwargs,
                power_meter_dt=power_meter_dt
            )

            voltage_array = channel_data[channel_to_save]

            if np.any(np.abs(voltage_array) > voltage_threshold_mV):
                print("")
                print("Please change gain")
                input("Press Enter to continue the scan...")

            print(f"Measured power during trace: {mean_power_uW:.3g} uW")

            voltage_path, time_path = save_arrays(
                voltage_array=voltage_array,
                time_array=time_array,
                commanded_power_mW=commanded_power_mW,
                measured_power_uW=mean_power_uW,
                iteration=iteration,
                base_directory=base_directory,
                save_time_once=True
            )

            print("Saved voltage:")
            print(voltage_path)

            if time_path is not None:
                print("Time file:")
                print(time_path)

            iteration += 1

    print("")
    print("Power scan complete.")


In [ ]:

# ---------------------------------------------------------------------
# Example use
# ---------------------------------------------------------------------

power_list = np.arange(10, 20, 1)/1000
n_iter = 1



log_file = run_power_scan(
    power_list=power_list,
    n_iterations=n_iter,
    laser=laser,
    powermeter=powermeter,
    base_directory='D:/Data',
    settle_time_s=1.0,
    power_meter_dt=0.02,
    channel_to_save=0,
    scope_kwargs=None
)